# Handout – Block 2: Linux-Grundlagen II

## Administration, Netzwerk, SSH und einfache Shell-Automatisierung

Block 2 erweitert die Grundlagen aus Block 1 um typische Aufgaben der Linux-Administration. Wir betrachten Paketmanagement, Prozesse, Dienste und Netzwerkzugriffe und verbinden anschließend vorhandene Kommandos mit einfachen Shell-Skripten.

Shell-Scripting dient hier bewusst **nicht** als Einführung in eine zweite Programmiersprache. Die Shell soll vor allem vorhandene Werkzeuge zu einem reproduzierbaren Ablauf verbinden. Eigene Programmlogik behandeln wir später mit Python.

### Lernziele

Nach diesem Block können Sie:

- das Grundprinzip eines Linux-Paketmanagers erklären,
- Pakete suchen, untersuchen, installieren und entfernen,
- Prozesse untersuchen und gezielt beenden,
- Dienste mit `systemctl` verwalten,
- Hostname, IP-Adresse, Route und Erreichbarkeit untersuchen,
- Host, IP-Adresse und Port unterscheiden,
- sich per SSH mit einem entfernten Linux-System verbinden,
- Passwort- und Public-Key-Authentifizierung unterscheiden,
- ein SSH-Schlüsselpaar einordnen,
- einfache Shell-Skripte aus Befehlssequenzen erstellen,
- Variablen, Exit Codes, `&&` und einfache Bedingungen verwenden,
- die Rollen von Shell, Python und Ansible voneinander abgrenzen.

> Einige administrative und netzwerkabhängige Beispiele können nur in der vorbereiteten Trainingsumgebung vollständig ausgeführt werden.

# 1. Paketmanagement

Software wird unter Linux typischerweise über einen **Paketmanager** installiert und verwaltet.

Ein Paket enthält nicht nur Programmdateien, sondern auch Metadaten, beispielsweise Name, Version und Abhängigkeiten.

Der Paketmanager arbeitet mit konfigurierten **Repositories**:

```text
Repository
    │
    │ Paketinformationen + Pakete
    ▼
Paketmanager
    │
    ▼
Linux-System
```

Je nach Distribution kommen unterschiedliche Werkzeuge zum Einsatz:

| Distributionen | Paketmanager |
|---|---|
| Debian, Ubuntu | `apt` |
| Fedora, RHEL, Rocky Linux | `dnf` |
| ältere RHEL/CentOS-Systeme | `yum` |

Im Seminar verwenden wir praktisch den Paketmanager der Trainingsumgebung. Die anderen Varianten müssen nicht parallel geübt werden.

In [1]:
%%bash
echo "Verfügbare Paketmanager:"
for cmd in apt dnf yum; do
    if command -v "$cmd" >/dev/null 2>&1; then
        echo "  $cmd -> $(command -v "$cmd")"
    fi
done

Verfügbare Paketmanager:
  apt -> /usr/bin/apt


## Paketinformationen aktualisieren

Bei Debian/Ubuntu aktualisiert

```text
apt update
```

die **Paketinformationen** aus den konfigurierten Repositories.

Das ist noch kein Upgrade der installierten Software.

Da `apt update` administrative Rechte und Netzwerkzugriff benötigt, führen wir es nicht automatisch im Handout aus.

Typischer Aufruf:

```text
sudo apt update
```

## Pakete suchen und untersuchen

Diese Operationen verändern das System nicht und können auf einem Debian-/Ubuntu-System direkt ausprobiert werden.

In [ ]:
%%bash
if command -v apt >/dev/null 2>&1; then
    apt search tree 2>/dev/null | head -n 12
else
    echo "Dieses System verwendet nicht apt."
fi

In [ ]:
%%bash
if command -v apt >/dev/null 2>&1; then
    apt show tree 2>/dev/null | head -n 18
else
    echo "Dieses System verwendet nicht apt."
fi

## Pakete installieren und entfernen

Auf einem Debian-/Ubuntu-System lauten typische Befehle:

```text
sudo apt install tree
sudo apt remove tree
```

Entsprechend gibt es bei `dnf` beispielsweise:

```text
sudo dnf install tree
sudo dnf remove tree
```

Wichtig ist die Trennung:

**Paket installieren ≠ Dienst starten.**

Ein installiertes Paket kann ein Kommando, Bibliotheken, Konfigurationsdateien oder auch einen Dienst enthalten.

# 2. Programme und Prozesse

Ein **Programm** ist ausführbarer Code auf einem Datenträger. Wird das Programm gestartet, entsteht ein **Prozess**.

```text
Programm
   │ starten
   ▼
Prozess
   │
   ├── PID
   ├── Benutzer
   ├── Zustand
   └── Ressourcen
```

Jeder Prozess besitzt eine **Process ID (PID)**.

In [ ]:
%%bash
echo "Eigene Shell:"
echo "PID der aktuellen Bash: $$"

echo
echo "Einige laufende Prozesse:"
ps -ef | head

`ps` zeigt Prozesse. Eine häufig verwendete Variante ist `ps aux`.

In [2]:
%%bash
ps aux | head

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root           1  0.0  0.0 167216 12688 ?        Ss   Aug28   0:02 /sbin/init splash
root           2  0.0  0.0      0     0 ?        S    Aug28   0:00 [kthreadd]
root           3  0.0  0.0      0     0 ?        S    Aug28   0:00 [pool_workqueue_release]
root           4  0.0  0.0      0     0 ?        I<   Aug28   0:00 [kworker/R-rcu_g]
root           5  0.0  0.0      0     0 ?        I<   Aug28   0:00 [kworker/R-rcu_p]
root           6  0.0  0.0      0     0 ?        I<   Aug28   0:00 [kworker/R-slub_]
root           7  0.0  0.0      0     0 ?        I<   Aug28   0:00 [kworker/R-netns]
root           9  0.0  0.0      0     0 ?        I<   Aug28   0:00 [kworker/0:0H-events_highpri]
root          12  0.0  0.0      0     0 ?        I<   Aug28   0:00 [kworker/R-mm_pe]


## Einen Prozess finden

`pgrep` sucht Prozesse nach ihrem Namen.

Für das Beispiel starten wir einen ungefährlichen `sleep`-Prozess.

In [ ]:
%%bash
sleep 300 &
PID=$!

echo "Gestartete PID: $PID"
echo "Prozess:"
ps -p "$PID" -o pid,ppid,user,stat,cmd

kill "$PID"
wait "$PID" 2>/dev/null || true

## Prozesse beenden

Das Kommando

```text
kill PID
```

sendet standardmäßig ein Signal an einen Prozess. Der Name `kill` bedeutet daher nicht automatisch „sofort gewaltsam beenden“.

`kill -9` erzwingt eine Beendigung und sollte **nicht** als Standardlösung verwendet werden. Ein Prozess sollte zunächst die Möglichkeit erhalten, regulär zu reagieren und aufzuräumen.

# 3. Dienste und systemd

Viele Serverprogramme laufen dauerhaft im Hintergrund als **Dienste**.

Auf vielen aktuellen Linux-Systemen verwaltet **systemd** solche Dienste. Eine Service Unit trägt typischerweise einen Namen wie:

```text
ssh.service
cron.service
nginx.service
```

Das zentrale Werkzeug ist `systemctl`.

In [ ]:
%%bash
if command -v systemctl >/dev/null 2>&1; then
    systemctl --version | head -n 2
else
    echo "systemctl ist in dieser Umgebung nicht vorhanden."
fi

## Laufender Zustand und Systemstart

Zwei Fragen müssen getrennt werden:

1. **Läuft der Dienst jetzt?**
2. **Soll der Dienst beim Systemstart automatisch gestartet werden?**

Daraus ergeben sich unterschiedliche Operationen:

| Kommando | Bedeutung |
|---|---|
| `systemctl status NAME` | Status anzeigen |
| `systemctl is-active NAME` | aktuellen Laufzustand prüfen |
| `systemctl start NAME` | jetzt starten |
| `systemctl stop NAME` | jetzt stoppen |
| `systemctl restart NAME` | neu starten |
| `systemctl reload NAME` | Konfiguration neu laden, falls unterstützt |
| `systemctl is-enabled NAME` | Autostart prüfen |
| `systemctl enable NAME` | Autostart aktivieren |
| `systemctl disable NAME` | Autostart deaktivieren |

**`start` und `enable` sind nicht dasselbe.**

In [ ]:
%%bash
if command -v systemctl >/dev/null 2>&1; then
    echo "Beispiel vorhandener Service Units:"
    systemctl list-unit-files --type=service --no-pager 2>/dev/null | head -n 12
else
    echo "systemctl ist nicht verfügbar."
fi

Statusabfragen sind meist ungefährlich. Start, Stop, Restart, Enable und Disable benötigen dagegen häufig administrative Rechte und werden in der Trainingsumgebung an einem vorgegebenen Dienst geübt.

Typische Aufrufe:

```text
systemctl status ssh
systemctl is-active ssh
systemctl is-enabled ssh

sudo systemctl restart ssh
```

# 4. Netzwerkgrundlagen

Für Remote-Administration und Ansible benötigen wir nur einige grundlegende Netzwerkbegriffe.

## Hostname

Der **Hostname** ist der Name eines Systems.

In [ ]:
%%bash
hostname

## IP-Adresse

Eine IP-Adresse identifiziert eine Netzwerkschnittstelle in einem IP-Netz.

Auf Linux-Systemen können Adressen beispielsweise so angezeigt werden:

In [3]:
%%bash
echo "hostname -I:"
hostname -I 2>/dev/null || true

echo
echo "ip addr:"
if command -v ip >/dev/null 2>&1; then
    ip addr | head -n 30
else
    echo "Das Kommando ip ist nicht installiert."
fi

hostname -I:
192.168.178.100 192.168.178.121 172.17.0.1 172.39.1.5 fd9a:b201:1642:0:231c:5137:ba64:a53d fd9a:b201:1642:0:8ad6:d88e:1d64:d0e3 2001:9e8:a5a6:5700:802f:53de:9b3c:4d61 2001:9e8:a5a6:5700:687a:71c4:7952:6c41 fd9a:b201:1642:0:c999:6fbd:745:901d fd9a:b201:1642:0:17a3:1189:7196:49f 2001:9e8:a5a6:5700:f688:279d:bf70:a166 2001:9e8:a5a6:5700:5062:94a7:f6e6:5e3e 

ip addr:
1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eno1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UP group default qlen 1000
    link/ether 04:0e:3c:c5:db:95 brd ff:ff:ff:ff:ff:ff
    altname enp16s0
    inet 192.168.178.100/24 brd 192.168.178.255 scope global dynamic noprefixroute eno1
       valid_lft 862682sec preferred_lft 862682s

## Routing

Die Routingtabelle entscheidet, über welchen Weg Zielnetze erreicht werden. Besonders wichtig ist häufig die **Default Route**.

In [4]:
%%bash
if command -v ip >/dev/null 2>&1; then
    ip route
else
    echo "Das Kommando ip ist nicht installiert."
fi

default via 192.168.178.1 dev eno1 proto dhcp metric 100 
default via 192.168.178.1 dev wlo1 proto dhcp metric 600 
169.254.0.0/16 dev wlo1 scope link metric 1000 
172.17.0.0/16 dev br-7b0c50b7d253 proto kernel scope link src 172.17.0.1 linkdown 
172.39.1.0/24 dev docker0 proto kernel scope link src 172.39.1.5 linkdown 
192.168.178.0/24 dev eno1 proto kernel scope link src 192.168.178.100 metric 100 
192.168.178.0/24 dev wlo1 proto kernel scope link src 192.168.178.121 metric 600 


## Ports

Eine IP-Adresse identifiziert ein System bzw. eine Netzwerkschnittstelle. Ein **Port** identifiziert einen Kommunikationsendpunkt eines Dienstes.

Beispiele:

```text
Server: 192.0.2.10
SSH:    192.0.2.10:22
HTTP:   192.0.2.10:80
HTTPS:  192.0.2.10:443
```

Daher sind **PID und Portnummer völlig unterschiedliche Dinge**.

In [ ]:
%%bash
if command -v ss >/dev/null 2>&1; then
    echo "Einige lokale Listening-Sockets:"
    ss -lnt | head
else
    echo "ss ist in dieser Umgebung nicht installiert."
fi

## Erreichbarkeit mit `ping`

`ping` verwendet ICMP, um die Erreichbarkeit eines Systems zu untersuchen.

Wichtig:

> Ein erfolgreicher `ping` beweist nicht, dass beispielsweise SSH oder HTTP funktioniert.

Ebenso kann ein System ICMP blockieren und trotzdem einen erreichbaren Anwendungsdienst anbieten.

In [ ]:
%%bash
ping -c 2 127.0.0.1

# 5. SSH – Remote-Zugriff

**SSH (Secure Shell)** ermöglicht eine verschlüsselte Verbindung zu einem entfernten System.

Beteiligte Komponenten:

```text
lokaler Rechner                 Remote-System
┌───────────────┐              ┌───────────────┐
│ SSH-Client    │ ─── SSH ───▶ │ SSH-Server    │
└───────────────┘              └───────────────┘
```

Grundsyntax:

```text
ssh user@host
```

Beispiel:

```text
ssh training@node1
```

SSH verwendet standardmäßig TCP-Port **22**. Ein abweichender Port kann mit `-p` angegeben werden:

```text
ssh -p 2222 training@node1
```

Die tatsächliche Verbindung wird in der Trainingsumgebung mit einem bereitgestellten Zielsystem durchgeführt.

In [ ]:
%%bash
echo "SSH-Client:"
if command -v ssh >/dev/null 2>&1; then
    ssh -V 2>&1
else
    echo "ssh ist nicht installiert."
fi

## Lokal oder remote?

Nach einer SSH-Anmeldung arbeiten Sie in einer Shell auf dem entfernten Rechner.

Deshalb sollte man bewusst prüfen:

```text
whoami
hostname
```

Vor und nach der Anmeldung können Benutzername und Hostname unterschiedlich sein.

Eine interaktive SSH-Sitzung wird mit

```text
exit
```

beendet.

# 6. Host Keys

Beim ersten Kontakt mit einem SSH-Server kennt der Client dessen **Host Key** noch nicht.

Der Host Key dient dazu, die Identität des Servers zu prüfen. Nach einer bestätigten Verbindung wird die Information typischerweise in

```text
~/.ssh/known_hosts
```

gespeichert.

Eine Warnung über einen geänderten Host Key sollte nicht unreflektiert ignoriert werden. Sie kann beispielsweise auf einen neu installierten Server hinweisen, aber auch auf ein Sicherheitsproblem.

In [ ]:
%%bash
echo "SSH-Verzeichnis:"
ls -la ~/.ssh 2>/dev/null || echo "~/.ssh existiert noch nicht."

# 7. SSH-Keys und Public-Key-Authentifizierung

Bei der Public-Key-Authentifizierung wird ein **Schlüsselpaar** verwendet:

```text
privater Schlüssel       öffentlicher Schlüssel
      │                           │
      │ bleibt beim Client        │ darf verteilt werden
      ▼                           ▼
   Client                     Server
```

Der wichtigste Sicherheitsgrundsatz lautet:

> **Der private Schlüssel wird nicht auf den Zielserver kopiert oder weitergegeben.**

Der öffentliche Schlüssel wird auf dem Zielsystem typischerweise in

```text
~/.ssh/authorized_keys
```

hinterlegt.

## Ein Trainings-Schlüsselpaar erzeugen

Die folgende Zelle erzeugt bewusst einen separaten Schlüssel unter `/tmp`. Dadurch wird kein vorhandener persönlicher SSH-Key verändert.

In [ ]:
%%bash
KEYDIR=/tmp/linux-handout-block2-ssh
rm -rf "$KEYDIR"
mkdir -p "$KEYDIR"

ssh-keygen -q -t ed25519 -N "" -f "$KEYDIR/training_key"

ls -l "$KEYDIR"

Die Datei ohne `.pub` ist der **private Schlüssel**. Die Datei mit `.pub` enthält den **öffentlichen Schlüssel**.

In [ ]:
%%bash
KEYDIR=/tmp/linux-handout-block2-ssh

echo "Öffentlicher Schlüssel:"
cat "$KEYDIR/training_key.pub"

echo
echo "Fingerabdruck:"
ssh-keygen -lf "$KEYDIR/training_key.pub"

In einer realen Trainingsverbindung kann der Public Key beispielsweise mit

```text
ssh-copy-id training@node1
```

installiert werden.

Danach kann sich der Client mit seinem privaten Schlüssel authentifizieren.

Diese SSH-Infrastruktur ist später eine direkte Grundlage für Ansible.

# 8. Einfache Shell-Skripte

Ein Shell-Skript ist eine Textdatei mit Shell-Kommandos.

Für diesen Kurs ist die Rolle bewusst begrenzt:

```text
Shell
  → vorhandene Programme zu Abläufen verbinden

Python
  → eigene Programmlogik implementieren

Ansible
  → Zustände auf Systemen automatisiert herstellen
```

**Ansible setzt nicht voraus, dass Sie eigene Python-Skripte schreiben.**

Python wird später trotzdem behandelt, damit Sie Skripte verstehen und eigene Automatisierungslogik entwickeln können.

## Shebang

Die erste Zeile eines Skripts kann festlegen, welcher Interpreter verwendet werden soll:

```text
#!/usr/bin/env bash
```

Danach folgen normale Shell-Kommandos.

In [5]:
%%bash
SCRIPT=/tmp/linux-handout-block2-hello.sh

cat > "$SCRIPT" <<'EOF'
#!/usr/bin/env bash

echo "Hallo aus dem Shell-Skript"
hostname
EOF

echo "Inhalt:"
cat "$SCRIPT"

echo
echo "Ausführung über Bash:"
bash "$SCRIPT"

Inhalt:
#!/usr/bin/env bash

echo "Hallo aus dem Shell-Skript"
hostname

Ausführung über Bash:
Hallo aus dem Shell-Skript
sophie-HP-Pavilion-Laptop-15-cs3xxx


## Skript direkt ausführen

Damit eine Datei direkt gestartet werden kann, benötigt sie das Execute-Bit.

In [ ]:
%%bash
SCRIPT=/tmp/linux-handout-block2-hello.sh

chmod u+x "$SCRIPT"
ls -l "$SCRIPT"

echo
echo "Direkte Ausführung:"
"$SCRIPT"

# 9. Variablen in einfachen Shell-Skripten

Eine Variable wird ohne Leerzeichen um `=` gesetzt:

```text
NAME=Wert
```

Beim Lesen wird `$NAME` verwendet.

Variablen sollten in vielen Fällen quotiert werden:

```text
"$NAME"
```

In [ ]:
%%bash
SCRIPT=/tmp/linux-handout-block2-vars.sh

cat > "$SCRIPT" <<'EOF'
#!/usr/bin/env bash

TARGET="training server"
echo "Ziel: $TARGET"
EOF

bash "$SCRIPT"

# 10. Exit Codes

Jedes ausgeführte Programm liefert einen **Exit Code** zurück.

Grundkonvention:

```text
0       Erfolg
ungleich 0   Fehler oder besonderer Zustand
```

Der Exit Code des zuletzt ausgeführten Kommandos steht in `$?`.

In [ ]:
%%bash
true
echo "Exit Code von true: $?"

false
echo "Exit Code von false: $?"

`$?` muss unmittelbar nach dem interessierenden Kommando ausgewertet werden. Jeder weitere Befehl überschreibt den gespeicherten Status.

In [ ]:
%%bash
ls /tmp >/dev/null
STATUS=$?

echo "Gespeicherter Exit Code: $STATUS"

# 11. Schritte nur bei Erfolg fortsetzen

Mit `&&` wird der nächste Befehl nur ausgeführt, wenn der vorherige erfolgreich war.

In [ ]:
%%bash
echo "Erfolgreiche Kette:"
true && echo "Schritt 2 wird ausgeführt"

echo
echo "Fehlschlagende Kette:"
false && echo "Diese Ausgabe erscheint nicht"

echo "Das Skript läuft danach weiter."

Für etwas besser lesbare Abläufe kann eine einfache `if`-Bedingung verwendet werden.

In [ ]:
%%bash
if test -d /tmp; then
    echo "/tmp ist vorhanden."
else
    echo "/tmp ist nicht vorhanden."
fi

Ein Kommando kann direkt als Bedingung verwendet werden:

```text
if command; then
    ...
else
    ...
fi
```

Die Shell wertet dabei den Exit Code aus.

In [ ]:
%%bash
if grep -q "root" /etc/passwd; then
    echo "Der Suchbegriff wurde gefunden."
else
    echo "Der Suchbegriff wurde nicht gefunden."
fi

# 12. Shell als Orchestrierung

Der zentrale Anwendungsfall in diesem Seminar ist nicht komplexe Bash-Programmierung, sondern eine **Sequenz vorhandener Werkzeuge**.

Das Muster lautet:

```text
Werkzeug A
    │
    ├── Fehler → abbrechen / melden
    │
    ▼
Werkzeug B
    │
    ├── Fehler → abbrechen / melden
    │
    ▼
Werkzeug C
```

Ein späterer Ablauf könnte konzeptionell beispielsweise so aussehen:

```text
git ...
python my_script.py
ping ...
```

Das konkrete Pipeline-Beispiel gehört nicht in den Lernpfad, ist aber als Anwendungsmuster für Shell-Scripting wichtig.

Die folgende ausführbare Variante verwendet nur lokale und ungefährliche Kommandos, demonstriert aber dasselbe Prinzip.

In [ ]:
%%bash
SCRIPT=/tmp/linux-handout-block2-pipeline.sh

cat > "$SCRIPT" <<'EOF'
#!/usr/bin/env bash

WORK=/tmp/linux-handout-block2-pipeline
mkdir -p "$WORK"

echo "Schritt 1: Datei vorbereiten"
echo "data" > "$WORK/input.txt"

if test -f "$WORK/input.txt"; then
    echo "Schritt 2: Datei verarbeiten"
    wc -c "$WORK/input.txt"

    echo "Schritt 3: Ergebnis prüfen"
    test -s "$WORK/input.txt"
else
    echo "Fehler: Eingabedatei fehlt."
    exit 1
fi

echo "Ablauf erfolgreich beendet."
EOF

chmod u+x "$SCRIPT"
"$SCRIPT"

# 13. Was gehört wohin?

Für den weiteren Seminarverlauf hilft folgende Abgrenzung:

| Werkzeug | Schwerpunkt |
|---|---|
| Shell | vorhandene Kommandos und Programme verbinden |
| Python | eigene Programmlogik und Automatisierungsskripte |
| Ansible | gewünschte Zustände auf einem oder vielen Systemen herstellen |

Diese Grenzen sind technisch nicht absolut. Sie sind aber ein gutes **didaktisches Modell**.

Insbesondere gilt:

> **Ansible kann vollständig sinnvoll eingesetzt werden, ohne dass Anwender eigene Python-Skripte schreiben.**

Python ist für uns ein zusätzliches Automatisierungswerkzeug und hilft später außerdem beim Verständnis des Ansible-Ökosystems.

# 14. Kommandos im Überblick

| Kommando | Zweck |
|---|---|
| `apt`, `dnf`, `yum` | Pakete verwalten |
| `ps` | Prozesse anzeigen |
| `pgrep` | Prozesse nach Namen suchen |
| `kill` | Signal an Prozess senden |
| `systemctl` | systemd-Units verwalten |
| `hostname` | Hostname anzeigen |
| `hostname -I` | IP-Adressen kompakt anzeigen |
| `ip addr` | Netzwerkschnittstellen und Adressen |
| `ip route` | Routingtabelle |
| `ping` | IP-Erreichbarkeit untersuchen |
| `ss` | Sockets/Ports untersuchen |
| `ssh` | Remote-Verbindung |
| `ssh-keygen` | SSH-Schlüsselpaar erzeugen |
| `ssh-copy-id` | Public Key auf Zielsystem bereitstellen |
| `chmod` | u. a. Skript ausführbar machen |

Wichtige Shell-Elemente:

| Element | Bedeutung |
|---|---|
| `#!/usr/bin/env bash` | Shebang |
| `NAME=Wert` | Variable setzen |
| `"$NAME"` | Variable verwenden |
| `$?` | Exit Code des letzten Kommandos |
| `&&` | nur bei Erfolg fortsetzen |
| `if ...; then ... fi` | einfache Bedingung |
| `exit N` | Skript mit Exit Code beenden |

# 15. Selbstkontrolle

1. Was ist die Aufgabe eines Paketmanagers?
2. Was unterscheidet `apt update` von einer Paketinstallation oder einem Upgrade?
3. Was unterscheidet ein Programm von einem Prozess?
4. Was ist eine PID?
5. Warum sollte `kill -9` nicht die Standardmethode zum Beenden eines Prozesses sein?
6. Was unterscheidet `systemctl start` von `systemctl enable`?
7. Was unterscheidet eine IP-Adresse von einem Port?
8. Warum beweist ein erfolgreicher `ping` nicht, dass SSH funktioniert?
9. Welche Rollen haben SSH-Client und SSH-Server?
10. Welcher Teil eines SSH-Schlüsselpaars darf auf den Server kopiert werden?
11. Was bedeutet Exit Code `0`?
12. Warum muss `$?` unmittelbar nach dem relevanten Kommando ausgewertet werden?
13. Was bewirkt `&&`?
14. Welche Rolle soll Shell-Scripting in diesem Seminar hauptsächlich übernehmen?
15. Muss man eigene Python-Skripte schreiben, um Ansible sinnvoll einsetzen zu können?

# 16. Überleitung zu Block 3

Sie können nun typische Linux-Aufgaben lokal durchführen, ein entferntes System per SSH erreichen und einfache Abläufe mit Shell-Skripten reproduzierbar machen.

Bei mehreren Zielsystemen entstehen jedoch neue Fragen:

- Wie führe ich dieselbe Aufgabe auf vielen Hosts aus?
- Wie erkenne ich unterschiedliche Ausgangszustände?
- Wie stelle ich einen gewünschten Zustand reproduzierbar her?
- Wie verwalte ich Konfiguration zentral?
- Wie behalte ich Fehler auf einzelnen Hosts im Blick?

Damit ergibt sich die Leitfrage für Block 3:

> **Wie können administrative Aufgaben zentral, wiederholbar und auf mehrere Linux-Systeme verteilt automatisiert werden?**

Hier setzt Ansible an.